In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA gold;


#Transformation Logic

In [0]:
df_fact_sales = spark.sql("""
SELECT
    s.order_number,
    s.order_date,
    s.customer_id,
    p.product_id,
    s.quantity,
    s.price,
    s.quantity * s.price AS sales_amount
FROM silver.crm_sales_details s
JOIN gold.dim_customers c
    ON s.customer_id = c.customer_id
LEFT JOIN gold.dim_products p
    ON TRIM(UPPER(s.product_number)) = TRIM(UPPER(p.product_key))
""")


In [0]:
df_fact_sales.display()


#Writing Gold Table

In [0]:
df_fact_sales.write \
    .mode("overwrite") \
    .saveAsTable("gold.fact_sales")


In [0]:
%sql
select * from workspace.gold.fact_sales

#Sanity Checks

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM gold.fact_sales;


#Sanity Checks

In [0]:
%sql
SELECT COUNT(*) AS null_customers
FROM gold.fact_sales
WHERE customer_id IS NULL;


In [0]:
%sql
COMMENT ON TABLE workspace.gold.fact_sales IS
'Sales fact table containing transactional sales data linked to customer, product, and date dimensions.';
